In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("yeast.csv")

In [4]:
X = df.drop("name", axis=1)
y = df["name"]

In [6]:
# again merging all small frequencies into OTHERS or any variable...
merge = {
    "ME1": "OTHER",
    "EXC": "OTHER",
    "VAC": "OTHER",
    "POX": "OTHER",
    "ERL": "OTHER"
}
y = y.replace(merge)

In [8]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y)
n_classes = len(le.classes_)

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

In [14]:
from sklearn.tree import DecisionTreeClassifier
class AdaBoostSAMME:
    def __init__(self, n_estimators=100):
        self.n_estimators = n_estimators
        self.models = []
        self.alphas = []
    def fit(self, X, y):
        n_samples = X.shape[0]
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        w = np.ones(n_samples) / n_samples

        for _ in range(self.n_estimators):
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y, sample_weight=w)
            pred = stump.predict(X)
            incorrect = (pred != y)
            err = np.sum(w * incorrect)
            if err <= 0:
                break
                
            alpha = np.log((1 - err) / err) + np.log(n_classes - 1)
            
            w *= np.exp(alpha * incorrect)
            w /= np.sum(w)
            self.models.append(stump)
            self.alphas.append(alpha)

    def predict(self, X):
        n_samples = X.shape[0]
        n_classes = len(self.classes)
        class_scores = np.zeros((n_samples, n_classes))

        for model, alpha in zip(self.models, self.alphas):
            preds = model.predict(X)
            for i in range(n_samples):
                class_scores[i, preds[i]] += alpha

        return np.argmax(class_scores, axis=1)

In [15]:
ada = AdaBoostSAMME(n_estimators=100)
ada.fit(X_train.values, y_train)

In [16]:
pred = ada.predict(X_test.values)

In [17]:
from sklearn.metrics import classification_report
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.50      0.68      0.58        93
           1       0.26      0.60      0.36        10
           2       0.79      0.84      0.82        32
           3       0.59      0.47      0.52        49
           4       0.60      0.45      0.52        86
           5       0.50      0.19      0.27        27

    accuracy                           0.55       297
   macro avg       0.54      0.54      0.51       297
weighted avg       0.57      0.55      0.54       297

